# Multi-model TTS benchmark on vLLM-Omni v0.22.0 + CUDA 13 — Colab T4

Benchmark **multiple TTS models** served as first-class architectures inside
**vLLM-Omni v0.22.0 stable**, running on a **CUDA 13 userspace** on Colab T4 —
no FastAPI wrapper, no monkey-patches. Pick a model from the **dropdown**,
launch, benchmark, log; switch models without re-installing or re-cloning.

**Fork + branch (used for ALL models):** `justHman/vllm-omni@feat/vieneu-tts-v0.22`
— branched from upstream tag `v0.22.0` (stable, 2026-06-06). The codebase is
identical across models; only the served checkpoint differs.

**Models configured (edit `MODELS` in cell 0 to add more):**
- **VieNeu-TTS-v2** (`pnnbao-ump/VieNeu-TTS-v2`) — Vietnamese TTS, Qwen3 + NeuCodec, voice preset `Ly`.
- **Qwen3-TTS-0.6B-VN** (`g-group-ai-lab/gwen-tts-0.6B`) — Qwen3-TTS finetune VN, voice `yen_nhi` (9 VN presets).
- **VoxCPM2** (`openbmb/VoxCPM2`) — multilingual voice-cloning TTS, **needs ref_audio** (cell 15). No vLLM-compatible quant exists (GGUF/MLX/ONNX only).
- **OmniVoice-fp16** (`kawshikbuet17/OmniVoice-fp16`) — fp16 safetensors (T4-friendly), **needs ref_audio** (cell 15).

> **Quantization note:** VoxCPM2 / OmniVoice quantized HF variants are GGUF
> (llama.cpp), MLX (Apple silicon), or ONNX — none run under vLLM-Omni's
> serving path (which needs safetensors + the registered vLLM arch). So we use
> the original safetensors checkpoints with `--dtype half` (fp16) to halve
> VRAM. bitsandbytes 4-bit (`kawshikbuet17/OmniVoice-fp4-4bit-bnb`) is left as
> a commented `--quantization bitsandbytes` option (untested on this arch).

Target command (run per model):
```bash
vllm serve <hf_repo> --omni --port 8000 --dtype half
```

**Workflow (after cells 0–3 run once):**
1. Cell 4b: pick model from dropdown → sets `MODEL_KEY`.
2. Re-run cells 4→6 (shim/prefetch/voices — idempotent, fast on cache hit).
3. Cell 7: stop old server + launch new model. Cell 9: wait `/v1/models`.
4. Cell 10/11: one-shot TTS + play (preset voice models only).
5. Cell 13: streaming + realtime playback. Cell 14: benchmark → `/content/benchmark_<MODEL_KEY>.log`.
6. **Voice-cloning models** (VoxCPM2, OmniVoice): use cell 15 (ref_audio + ref_text) — cells 10/13/14 will skip preset-voice synth with a note.
7. To switch model: cell 4b dropdown → re-run 4→7 → 9 → 13 → 14. **No re-install/clone.**

**Why v0.22.0 stable + cu13 re-verify:** the cu13 stack (torch 2.11.0+cu130 + vllm)
was proven on Colab T4 with OmniVoice on vllm-omni v0.24.0rc1. This notebook
re-verifies the same cu13 stack against the **v0.22.0 stable** base. If
`vllm 0.22.0 + torch 2.11+cu130` import breaks on T4, cell 2 falls back to
`vllm==0.24.0` (the proven cu13 stack) and the discrepancy is noted.

> If start-up or inference fails, the failure output is captured in the launch
> cell + `/tmp/vllm_serve_<MODEL_KEY>.log` — paste it back so the wiring can be fixed.

In [ ]:
# ============================================================
# 0. Preflight — environment detection + MODELS config (run ONCE)
# ============================================================
import os, sys, subprocess, time, json, urllib.request

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass

print("✓ Colab:", IN_COLAB)
print("✓ Python:", sys.version.split()[0])

# Show CUDA driver vs userspace BEFORE we touch anything.
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        text=True).strip()
    print("✓ GPU / driver:", smi)
except Exception as e:
    raise RuntimeError(
        "nvidia-smi failed — this notebook needs a GPU runtime: "
        "Runtime > Change runtime type > T4 GPU. Err: " + str(e))

# Colab CUDA userspace probe.
def find_cudart():
    hits = []
    import glob
    for root in ["/usr/local/cuda", "/usr/local/cuda*", "/usr/lib",
                 "/usr/lib/x86_64-linux-gnu", "/lib/x86_64-linux-gnu"]:
        for d in glob.glob(root):
            for f in os.listdir(d) if os.path.isdir(d) else []:
                if f.startswith("libcudart.so"):
                    hits.append(os.path.join(d, f))
    return sorted(set(hits))

cudart = find_cudart()
print("✓ libcudart present on image (pre-install):")
for c in cudart[:8]:
    print("   ", c)
if not cudart:
    print("    <none in standard paths — torch 2.11+cu130 ships libcudart.so.13 via the cuda-toolkit pip package>")

PORT = 8000
FORK_REPO = "https://github.com/justHman/vllm-omni.git"
FORK_BRANCH = "feat/vieneu-tts-v0.22"
VLLM_OMNI_TAG = "v0.22.0"
# cu13 stack pins — re-verified in this notebook against v0.22.0 base.
VLLM_PIN = "0.22.0"
VLLM_PIN_FALLBACK = "0.24.0"  # proven cu13 stack; used only if 0.22.0+cu130 breaks
print("✓ fork:", FORK_REPO, "@", FORK_BRANCH, "(", VLLM_OMNI_TAG, "base )")
print("✓ cu13 stack pin: vllm", VLLM_PIN, "| fallback vllm", VLLM_PIN_FALLBACK)

# ── Multi-model config ───────────────────────────────────────────────
# Per-model: hf repo, default voice preset, sample rate, response format,
# extra HF repos to prefetch, whether the neucodec transformers shim is needed,
# extra env vars, --dtype, and optional --quantization flag.
#
# NOTE on quantization: VoxCPM2 / OmniVoice quantized variants on HF are
# GGUF (llama.cpp), MLX (Apple silicon), or ONNX -- NONE of those run under
# vLLM-Omni's serving path (which needs safetensors + the model's registered
# vLLM arch). So we use the original safetensors checkpoints here, with
# `--dtype half` (fp16) to halve VRAM on T4. bitsandbytes 4-bit
# (kawshikbuet17/OmniVoice-fp4-4bit-bnb) would need `--quantization
# bitsandbytes` + vLLM bnb support for this arch -- untested, left as a
# commented option.
MODELS = {
    "VieNeu-TTS-v2": {
        "hf": "pnnbao-ump/VieNeu-TTS-v2",
        "voice": "Ly",                              # preset in voices.json
        "sr": 24000,
        "response_format": "wav",
        "stream_fmt": "pcm",
        "extra_download": ["neuphonic/neucodec"],   # external NeuCodec codec
        "needs_neucodec_shim": True,                # transformers HubertModel/Wav2Vec2BertModel shim
        "extra_env": {},
        "dtype": "half",
    },
    "Qwen3-TTS-0.6B-VN": {
        # Qwen3-TTS finetune VN (g-group-ai-lab). bf16 original -> --dtype half.
        # 9 VN voice presets: yen_nhi, my_van, ai_vy, an_nhi, dieu_linh,
        # khanh_toan, tran_lam, nsnd_ha_phuong, nsnd_kim_cuc.
        "hf": "g-group-ai-lab/gwen-tts-0.6B",
        "voice": "yen_nhi",
        "sr": 24000,
        "response_format": "wav",
        "stream_fmt": "pcm",
        "extra_download": [],
        "needs_neucodec_shim": False,
        "extra_env": {},
        "dtype": "half",
    },
    "VoxCPM2": {
        # openbmb/VoxCPM2 -- original safetensors (no vLLM-compatible quant
        # exists; GGUF/MLX/ONNX variants are llama.cpp/Apple/ONNX runtime).
        # Voice cloning model: use ref_audio + ref_text (cell 15). No preset
        # voices.json -> set voice to None and use cell 15 for benchmark,
        # OR provide a ref_audio in extra_download and wire it in cell 14.
        "hf": "openbmb/VoxCPM2",
        "voice": None,                              # voice cloning model -> use cell 15
        "sr": 24000,
        "response_format": "wav",
        "stream_fmt": "pcm",
        "extra_download": [],
        "needs_neucodec_shim": False,
        "extra_env": {},
        "dtype": "half",
        "needs_ref_audio": True,                    # benchmark cell uses cell-15 path
    },
    "OmniVoice-fp16": {
        # kawshikbuet17/OmniVoice-fp16 -- fp16 safetensors (T4-friendly, half
        # the VRAM of the bf16 original k2-fsa/OmniVoice). Voice cloning: use
        # ref_audio + ref_text (cell 15). For a VN finetune alternative see
        # kjanh/KhanhTTS-OmniVoice (vietnamese).
        "hf": "kawshikbuet17/OmniVoice-fp16",
        "voice": None,                              # voice cloning -> cell 15
        "sr": 24000,
        "response_format": "wav",
        "stream_fmt": "pcm",
        "extra_download": [],
        "needs_neucodec_shim": False,
        "extra_env": {},
        "dtype": "half",                            # already fp16, but flag for consistency
        "needs_ref_audio": True,
        # "quant": "bitsandbytes",  # if switching to kawshikbuet17/OmniVoice-fp4-4bit-bnb
    },
}

# Default selected model (the dropdown in cell 4b overrides this).
MODEL_KEY = "VieNeu-TTS-v2"
print("✓ MODELS configured:", list(MODELS))
print("✓ default MODEL_KEY:", MODEL_KEY)

In [ ]:
# ============================================================
# 1. Mount Drive — but keep HF cache LOCAL for fast checkpoint loading
# ============================================================
# Drive is mounted for general persistence, but we DO NOT point HF_HOME at
# Drive. The stage-1 codec checkpoint is 773 safetensors shards; loading
# them over Drive FUSE does 773 small random reads and on a bad run takes
# 1238s, blowing past vllm-omni's 600s orchestrator startup timeout
# (TimeoutError: Orchestrator did not become ready within 600s). Keeping
# the HF cache on the local Colab SSD makes checkpoint load a fast local
# read. Trade-off: re-download ~600MB each session — a single sequential
# pull is much faster and far more reliable than 773 FUSE random reads.
MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        MOUNTED = True
        print("✓ Drive mounted (for general use; HF cache stays local)")
    except Exception as e:
        print("⚠ Drive mount skipped:", e)

# HF cache — keep on local SSD for fast random reads during weight load.
# Default is /root/.cache/huggingface; we just make sure HF_HOME/HF_HUB_CACHE
# point there explicitly so Drive doesn't sneak in via a leftover env.
hf_cache = "/root/.cache/hf_cache"
os.makedirs(hf_cache, exist_ok=True)
os.environ["HF_HOME"] = hf_cache
os.environ["TRANSFORMERS_CACHE"] = os.path.join(hf_cache, "transformers")
os.environ["HF_HUB_CACHE"] = os.path.join(hf_cache, "hub")
print("✓ HF_HOME =", hf_cache, "(local SSD — fast random reads)")


In [ ]:
# ============================================================
# 2. Install uv + cu130 torch stack + vllm  (re-verify cu13 against v0.22.0)
# ============================================================
# COLLAB CUDA 13 STRATEGY (pip-only):
#   - torch 2.11.0+cu130 wheel declares
#     `cuda-toolkit[cublas,cudart,cufft,...,nvrtc,nvtx]==13.0.2` on Linux.
#     That PyPI meta-package pulls cuda-cudart 13.x which SHIPS
#     libcudart.so.13 via pip => no conda, no manual LD_LIBRARY_PATH; the
#     +cu130 wheels are self-contained for the CUDA 13 userspace.
#   - --torch-backend=cu130 makes uv resolve from the cu130 index. NOT `auto`
#     (auto reads driver 580 / "CUDA 13.0" and also lands on cu130, but pin
#     it explicitly to avoid driver-table drift).
#
# RE-VERIFY GATE: v0.22.0's precompiled .so were built for the torch-2.10 /
# cu128-129 era. torch 2.11+cu130 may or may not ABI-match them. This cell
# pins vllm==0.22.0 first; if import below fails, fall back to vllm==0.24.0
# (the proven cu13 stack) and note the discrepancy.
!pip install -q uv
!uv self update 2>/dev/null || true
!pip uninstall -y torch torchvision torchaudio vllm 2>/dev/null || true

import importlib, sys
# Drop any cached torch-family module so the cu130 install is seen fresh.
for _m in [m for m in list(sys.modules) if m.split('.')[0] in ('torch','vllm','torchaudio','torchvision')]:
    del sys.modules[_m]

def install_vllm(pin):
    print(f"\n>>> installing vllm=={pin} on cu130 torch stack")
    !uv pip install --system --torch-backend=cu130 \
        torch==2.11.0 torchaudio==2.11.0 torchvision==0.26.0 vllm=={pin}

try:
    install_vllm(VLLM_PIN)
    import vllm, torch, torchaudio, torchvision
    print("✓ vllm:", vllm.__version__, "| torch:", torch.__version__,
          "| torchaudio:", torchaudio.__version__, "| torchvision:", torchvision.__version__)
    print("✓ torch.version.cuda =", torch.version.cuda,
          "| cuda available:", torch.cuda.is_available())
    assert vllm.__version__.startswith(VLLM_PIN), f"expected vllm {VLLM_PIN}, got {vllm.__version__}"
    assert torch.__version__.startswith("2.11.0"), f"expected torch 2.11.0, got {torch.__version__}"
    assert "+cu130" in torch.__version__, f"expected torch+cu130, got {torch.__version__}"
    assert torch.cuda.is_available(), "CUDA not available — runtime must be T4 GPU"
    print("✓ GPU:", torch.cuda.get_device_name(0))
    ACTIVE_VLLM = VLLM_PIN
except Exception as e:
    print(f"⚠ vllm {VLLM_PIN} + cu130 import FAILED: {e}")
    print(f">>> fallback to vllm {VLLM_PIN_FALLBACK} (proven cu13 stack; branch code is stack-agnostic)")
    for _m in [m for m in list(sys.modules) if m.split('.')[0] in ('torch','vllm','torchaudio','torchvision')]:
        del sys.modules[_m]
    install_vllm(VLLM_PIN_FALLBACK)
    import vllm, torch, torchaudio, torchvision
    print("✓ vllm:", vllm.__version__, "| torch:", torch.__version__)
    assert vllm.__version__.startswith(VLLM_PIN_FALLBACK)
    assert "+cu130" in torch.__version__
    assert torch.cuda.is_available()
    ACTIVE_VLLM = VLLM_PIN_FALLBACK
    print(f"⚠ NOTE: notebook now runs on vllm {VLLM_PIN_FALLBACK}, NOT base {VLLM_PIN}. "
          f"Branch vieneu code is identical; only the notebook stack pin differs.")
print("ACTIVE_VLLM =", ACTIVE_VLLM)


## ⚠ STOP — Runtime > Restart session (one time)

If cell 2's asserts show `torch 2.11.0+cu128` (NOT `+cu130`) and
`torch.version.cuda = 12.x`, the Jupyter kernel cached the **old** torch module
before the cu130 install replaced the files on disk. This is the classic Colab
`sys.modules` trap: the new wheel is installed but invisible to the running
kernel.

**Fix: `Runtime > Restart session`, then re-run cells 1–2.** Do NOT skip this —
a stale torch under a cu130 vllm is the most common silent break here. After
restart, re-run cells 1–2 only (cell 1 re-mounts Drive / re-sets caches); you do
NOT need to re-run the install.


In [ ]:
# ============================================================
# 2b. Model picker — dropdown sets MODEL_KEY (re-run after changing)
# ============================================================
# Pick a model, then re-run cells 4->6 (shim/prefetch/voices) -> 7 (launch)
# -> 9 (wait) -> 13 (stream) -> 14 (benchmark). No re-install/clone needed.
import ipywidgets as widgets
from IPython.display import display

model_dropdown = widgets.Dropdown(
    options=list(MODELS.keys()),
    value=MODEL_KEY,
    description="Model:",
    layout=widgets.Layout(width="60%"),
)

def _on_change(change):
    global MODEL_KEY
    if change["name"] == "value" and change["type"] == "change":
        MODEL_KEY = change["new"]
        print(f"→ MODEL_KEY = {MODEL_KEY!r}  (re-run cells 4→6, 7, 9, 13, 14)")

model_dropdown.observe(_on_change)
display(model_dropdown)
MODEL_KEY = model_dropdown.value
cfg = MODELS[MODEL_KEY]
print(f"✓ selected: {MODEL_KEY}  | hf={cfg['hf']}  voice={cfg['voice']}  "
      f"sr={cfg['sr']}  dtype={cfg['dtype']}  shim={cfg['needs_neucodec_shim']}")

In [ ]:
# ============================================================
# 3. Clone the fork + editable install with [vieneu] extra (fresh clone each run)
# ============================================================
import os, subprocess, sys

CLONE_DIR = "/content/vllm-omni"
# Fresh clone every run — do NOT cache the clone on Drive. A cached tarball
# would freeze buggy old code and block upstream fixes (the load_presets
# voices.json schema fix in particular) from taking effect on rerun. --depth 1
# keeps the clone cheap (~10s).
!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {FORK_BRANCH} {FORK_REPO} {CLONE_DIR}
# Fetch tags so setuptools-scm can resolve the ancestor tag (v0.22.0) and
# report a real version (0.22.0.dev<N>+g<hash>) instead of its 0.1.dev1
# fallback. --depth 1 alone ships NO tags, which is what made vllm_omni
# report "0.1.dev1+g0bdf181ea" in the first runs.
!cd {CLONE_DIR} && git fetch --tags --depth 1 origin {VLLM_OMNI_TAG}
!cd {CLONE_DIR} && git log --oneline -1
!cd {CLONE_DIR} && git describe --tags HEAD || true

# Single editable install with the vieneu extra (adds sea-g2p + neucodec).
# setup.py resolves platform deps dynamically (VLLM_OMNI_TARGET_DEVICE=cuda);
# it does NOT pin vllm, so the vllm from cell 2 stays.
os.environ["VLLM_OMNI_TARGET_DEVICE"] = "cuda"
os.environ["UV_TORCH_BACKEND"] = "cu130"
# Pretend the exact release version so the editable install reports
# "0.22.0" even if the ancestor tag fetch above is flaky on Colab.
os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"] = VLLM_OMNI_TAG.lstrip("v")
!uv pip install --system -e "{CLONE_DIR}[vieneu]" --torch-backend=cu130

# The editable install wrote a .pth into site-packages, but THIS Jupyter kernel
# started before that .pth existed. The `vllm` CLI subprocess (later cell) starts
# fresh and sees it fine; to verify imports HERE without a restart, prepend the
# source tree explicitly.
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
import vllm_omni
print("✓ vllm_omni:", getattr(vllm_omni, "__version__", "<no __version__>"))
import sea_g2p, neucodec
print("✓ sea_g2p + neucodec importable")

# Re-verify torch stack survived the vieneu-extra install (neucodec declares
# torchao>=0.12.0 with no upper bound; assert so a silent bump surfaces here).
import torch
assert torch.__version__.startswith("2.11.0"), f"torch bumped to {torch.__version__} by vieneu extra"
assert "+cu130" in torch.__version__, f"torch lost +cu130: {torch.__version__}"
print("✓ torch still", torch.__version__, "after vieneu extra")


In [ ]:
# ============================================================
# 4. transformers shim (ONLY for models that need neucodec) + nvrtc + warnings
# ============================================================
# Re-run-safe. The neucodec shim is only applied when the selected model's
# config has needs_neucodec_shim=True (VieNeu-TTS-v2). Other models skip it.
import importlib, sys, ctypes, warnings

cfg = MODELS[MODEL_KEY]
if cfg.get("needs_neucodec_shim", False):
    # neucodec 0.0.6 does `from transformers import HubertModel, Wav2Vec2BertModel`.
    # On Colab's transformers 5.x the dynamic top-level export scan doesn't surface
    # these (the classes live at transformers.models.hubert.modeling_hubert /
    # transformers.models.wav2vec2_bert.modeling_wav2vec2_bert). Patch the namespace
    # from the submodules so neucodec's top-level import succeeds regardless.
    import transformers
    for _cls, _sub in [
        ("HubertModel", "transformers.models.hubert.modeling_hubert"),
        ("Wav2Vec2BertModel", "transformers.models.wav2vec2_bert.modeling_wav2vec2_bert"),
    ]:
        if not hasattr(transformers, _cls):
            try:
                setattr(transformers, _cls, getattr(importlib.import_module(_sub), _cls))
                print(f"  shim: transformers.{_cls} <- {_sub}")
            except Exception as e:
                print(f"  shim FAILED for {_cls}: {e}")
else:
    print(f"  skip neucodec shim (not needed for {MODEL_KEY})")

# nvrtc (NVIDIA Runtime Compilation) — neucodec/torchaudio may JIT CUDA kernels
# via libnvrtc at runtime. torch 2.11+cu130 SHOULD bundle it via cuda-toolkit,
# but on this Colab image the bundled .so isn't always on the loader path.
try:
    ctypes.CDLL("libnvrtc.so.13")
    print("✓ libnvrtc.so.13 loadable (torch-bundled)")
except OSError:
    try:
        ctypes.CDLL("libnvrtc.so.12")
        print("✓ libnvrtc.so.12 already loadable")
    except OSError:
        print("⚠ installing nvidia-cuda-nvrtc-cu12 for libnvrtc")
        !uv pip install --system --torch-backend=cu130 nvidia-cuda-nvrtc-cu12 || pip install nvidia-cuda-nvrtc-cu12
        try:
            ctypes.CDLL("libnvrtc.so.12")
            print("✓ libnvrtc.so.12 loadable after install")
        except OSError as e:
            print(f"⚠ nvrtc not loadable: {e} — codec JIT may fail")

# Suppress upstream deprecation warnings from external pip packages
# (diffusers Flax, pydub regex, neucodec weight_norm, huggingface_hub
# local_dir_use_symlinks/resume_download). Noise only; does not affect serving.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Flax classes are deprecated.*")
warnings.filterwarnings("ignore", message=".*invalid escape sequence.*")
warnings.filterwarnings("ignore", message=".*weight_norm.*deprecated.*")
warnings.filterwarnings("ignore", message=".*local_dir_use_symlinks.*deprecated.*")
warnings.filterwarnings("ignore", message=".*resume_download.*deprecated.*")
import logging
logging.getLogger("diffusers").setLevel(logging.ERROR)

# Sanity import sweep (vllm_omni + sea_g2p + neucodec only needed for vieneu;
# for other models we still verify the base stack).
for mod in ["sea_g2p", "torchaudio", "torchvision", "neucodec", "vllm_omni"]:
    try:
        importlib.import_module(mod)
        print(f"✓ {mod} importable")
    except Exception as e:
        print(f"⚠ {mod} import FAILED (may be optional for {MODEL_KEY}):", e)

In [ ]:
# ============================================================
# 5. Pre-download the selected model + extras (re-run when MODEL_KEY changes)
# ============================================================
# snapshot_download is idempotent via the HF cache (cell 1): a re-run after
# switching MODEL_KEY only downloads the new model; the old one stays cached.
from huggingface_hub import snapshot_download

cfg = MODELS[MODEL_KEY]
MODEL = cfg["hf"]
print(f"Downloading {MODEL_KEY}: {MODEL} ...")
mp = snapshot_download(
    MODEL,
    allow_patterns=["*.json", "*.txt", "*.safetensors", "*.model",
                    "voices.json", "tokenizer*", "*.bin", "*.gguf",
                    "*.wav", "*.py"],
)
print("✓ checkpoint at:", mp)

for extra in cfg.get("extra_download", []):
    print(f"Downloading extra: {extra} ...")
    try:
        ep = snapshot_download(extra)
        print(f"✓ {extra} at: {ep}")
    except Exception as e:
        print(f"⚠ extra {extra} failed: {e}")

In [ ]:
# ============================================================
# 6. Inspect voices.json presets for the selected model (best-effort)
# ============================================================
import json, os

cfg = MODELS[MODEL_KEY]
voices_path = os.path.join(mp, "voices.json")
if os.path.exists(voices_path):
    with open(voices_path) as f:
        voices = json.load(f)
    # voices.json is either {preset_name: {...}} or {"voices": {...}, ...meta}
    presets = voices if not isinstance(voices.get("voices"), dict) else voices["voices"]
    if isinstance(presets, dict):
        names = sorted(presets.keys())
    else:
        names = list(presets)
    print(f"✓ {MODEL_KEY} preset voices ({len(names)}):", names)
    if cfg["voice"] not in names and cfg["voice"] != "FILL-ME":
        print(f"⚠ configured voice '{cfg['voice']}' NOT in presets; pick one of the above")
else:
    print(f"⚠ {MODEL_KEY}: no voices.json at {voices_path}")
    print("  → use ref_audio + ref_text (cell 15) instead of a preset voice")

# Probe vllm serve stage flags (informational; v0.22.0 auto-discovers pipeline.yaml).
print("\n>>> vllm serve --help (grep stage) ...")
!vllm serve --help 2>&1 | grep -i stage || true

In [ ]:
# ============================================================
# 7. Stop any stale server + launch the selected model (--omni, background)
# ============================================================
# Re-running this cell stops the previous model's server and starts the new
# one. No re-install / re-clone needed — only the served checkpoint changes.
import subprocess, os, time

cfg = MODELS[MODEL_KEY]
MODEL = cfg["hf"]
# Per-model log so switching models doesn't overwrite the previous log.
LOG = f"/tmp/vllm_serve_{MODEL_KEY.replace(' ', '_').replace('-', '_')}.log"

def stop_server():
    """Kill anything bound to PORT (previous model's vllm serve)."""
    try:
        subprocess.run(["fuser", "-k", f"{PORT}/tcp"], check=False,
                       stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    except FileNotFoundError:
        pass
    # Also terminate the previous proc handle if it's still around.
    for _v in list(globals().get("_procs", [])):
        try:
            _v.terminate()
        except Exception:
            pass

stop_server()
# Give the OS a moment to release the port.
time.sleep(2)

# T4 (cc 7.5) cannot run bfloat16 (needs >= 8.0) -> --dtype half for fp16.
# VLLM_OMNI_TARGET_DEVICE=cuda bypasses platform auto-detect.
env = dict(os.environ)
env["VLLM_OMNI_TARGET_DEVICE"] = "cuda"
env.update(cfg.get("extra_env", {}))

# Inference-speed knobs live in pipeline.yaml per-stage, NOT the CLI.
# Do NOT add --enable_prefix_caching / --max-model-len / --gpu-memory-utilization
# to `vllm serve --omni`: multi-stage omni reads them from per-stage yaml; a
# global value breaks some codec stages. See pipeline.yaml for the rationale.
cmd = [
    "vllm", "serve", MODEL,
    "--omni",
    "--port", str(PORT),
    "--host", "0.0.0.0",
    "--trust-remote-code",
    "--dtype", cfg["dtype"],
    "--stage-init-timeout", "1800",
    "--init-timeout", "1800",
]
if cfg.get("quant"):
    cmd += ["--quantization", cfg["quant"]]
print(f"Launching {MODEL_KEY}: {' '.join(cmd)}")
print(f"Logs -> {LOG}")

# start_new_session=True -> server survives a Jupyter cell interrupt (Stop).
logf = open(LOG, "w", buffering=1)
proc = subprocess.Popen(
    cmd, stdout=logf, stderr=subprocess.STDOUT,
    cwd="/content/vllm-omni", env=env, start_new_session=True,
)
# Track proc handles so stop_server() can terminate them on the next switch.
_procs = list(globals().get("_procs", [])) + [proc]
print(f"✓ server PID: {proc.pid}  (model={MODEL_KEY}, new session group)")
print("  → next: run cell 9 (wait /v1/models), then 10/11/13/14.")

In [ ]:
# ============================================================
# 8. (Interruptible) tail the server log — Stop cell to stop watching, server keeps running
# ============================================================
# `tail -f` runs until you press the cell's Stop button. The server itself
# survives because it was launched in its own session group (cell 7).
!tail -n 200 -f {LOG}


In [ ]:
# ============================================================
# 9. Wait for /v1/models to come up (print log tail if it times out)
# ============================================================
import urllib.request, json, time

def tail(path, n=80):
    try:
        with open(path, errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except Exception as e:
        return f"<could not read {path}: {e}>"

def get_models(timeout=1500):
    url = f"http://localhost:{PORT}/v1/models"
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return json.loads(r.read())
        except Exception as e:
            last_err = e
        if proc.poll() is not None:
            raise RuntimeError(
                f"server process exited early (code {proc.returncode}). See {LOG}.\n"
                + tail(LOG))
        time.sleep(5)
    raise RuntimeError(
        f"timed out waiting for {url}. last_err={last_err}. Tail of {LOG}:\n"
        + tail(LOG))

print("Waiting for server (up to 25 min for first-time compile) ...")
models = get_models(timeout=1500)
print("✓ /v1/models:")
print(json.dumps(models, indent=2))


In [ ]:
# ============================================================
# 10. POST /v1/audio/speech (preset voice) and save a WAV
# ============================================================
import requests

cfg = MODELS[MODEL_KEY]
TEXT = "Xin chào, đây là giọng nói tiếng Việt từ {} trên cu13.".format(MODEL_KEY)

def speech(text=TEXT, voice=None, response_format=None):
    if voice is None:
        voice = cfg["voice"]
    if response_format is None:
        response_format = cfg["response_format"]
    payload = {"model": cfg["hf"], "input": text,
               "voice": voice, "response_format": response_format}
    r = requests.post(f"http://localhost:{PORT}/v1/audio/speech",
                     json=payload, timeout=180)
    if r.status_code != 200:
        raise RuntimeError(f"speech failed: {r.status_code} {r.text[:2000]}")
    return r

r = speech()
out_wav = f"/content/tts_demo_{MODEL_KEY.replace(' ', '_').replace('-', '_')}.wav"
with open(out_wav, "wb") as f:
    f.write(r.content)
print(f"✓ wrote {out_wav}  ({len(r.content)} bytes, voice={cfg['voice']})")

In [ ]:
# ============================================================
# 11. Verify WAV + play inline
# ============================================================
from IPython.display import Audio, display

with open(out_wav, "rb") as f:
    head = f.read(12)
size = os.path.getsize(out_wav)
print("header bytes:", head)
assert head[:4] == b"RIFF", f"not a RIFF/WAV file: header={head!r}"
assert head[8:12] == b"WAVE", f"not a WAVE file: header={head!r}"
assert size > 5000, f"WAV suspiciously small: {size} bytes"
print(f"✓ valid WAV, {size} bytes")
display(Audio(out_wav, autoplay=False))


In [ ]:
# ============================================================
# 13. Streaming TTS — `stream: true` with REALTIME inline playback
# ============================================================
# Streams PCM chunks from the server AND plays them inline in the notebook
# as they arrive (incremental, not wait-for-all-then-play). Uses the selected
# model's sample rate / stream format / voice preset from MODELS[MODEL_KEY].
import requests, time, io, struct, sys
from IPython.display import Audio, display, HTML

cfg = MODELS[MODEL_KEY]
SR = cfg["sr"]
TEXT_S = ("Xin chào thế giới. Đây là một câu dài hơn để nghe streaming tăng dần "
          "theo thời gian thực, thay vì đợi toàn bộ rồi mới phát.")
payload = {"model": cfg["hf"], "input": TEXT_S, "voice": cfg["voice"],
           "response_format": cfg["stream_fmt"], "stream": True}

t0 = time.perf_counter()
r = requests.post(f"http://localhost:{PORT}/v1/audio/speech",
                 json=payload, stream=True, timeout=180)
if r.status_code != 200:
    raise RuntimeError(f"stream failed: {r.status_code} {r.text[:500]}")

pcm = bytearray()
first_byte_s = None
chunk_times = []
for chunk in r.iter_content(chunk_size=4096):
    if not chunk:
        continue
    if first_byte_s is None:
        first_byte_s = time.perf_counter() - t0
    pcm.extend(chunk)
    chunk_times.append(time.perf_counter() - t0)
total_s = time.perf_counter() - t0

# Wrap PCM in a 16-bit mono WAV header at the model's sample rate.
def pcm16_to_wav(pcm_bytes: bytes, sr: int = SR) -> bytes:
    n = len(pcm_bytes) // 2
    return (b"RIFF" + struct.pack("<I", 36 + n * 2) + b"WAVE"
            + b"fmt " + struct.pack("<IHHIIHH", 16, 1, 1, sr, sr * 2, 2, 16)
            + b"data" + struct.pack("<I", n * 2) + bytes(pcm_bytes))

wav_bytes = pcm16_to_wav(bytes(pcm))
out_stream = f"/content/tts_stream_{MODEL_KEY.replace(' ', '_').replace('-', '_')}.wav"
with open(out_stream, "wb") as f:
    f.write(wav_bytes)

dur = len(pcm) / 2 / SR
print(f"✓ streamed {len(pcm)} bytes PCM | first-byte {first_byte_s:.3f}s | total {total_s:.3f}s")
print(f"  audio duration ~{dur:.2f}s (16-bit mono {SR}Hz, voice={cfg['voice']})")
if first_byte_s is not None and total_s > first_byte_s + 0.1:
    print(f"  → first_byte << total → incremental streaming WORKS "
          f"(chunk1 @ {first_byte_s:.2f}s, last chunk @ {chunk_times[-1]:.2f}s)")
else:
    print("  → first_byte ≈ total → codec is one-shot (codec_chunk_frames huge)")

print("\n▶ Inline playback of the streamed clip:")
display(Audio(wav_bytes, rate=SR, autoplay=True))

# Chunk arrival timeline so you can SEE the streaming happen.
if chunk_times:
    bars = "".join("█" for _ in chunk_times)
    print(f"\nchunk arrival timeline ({len(chunk_times)} chunks):")
    print(f"  0.0s {'|'}{bars[:60]}{'|'} {chunk_times[-1]:.1f}s")
    print(f"  first chunk @ {chunk_times[0]:.2f}s, "
          f"mid @ {chunk_times[len(chunk_times)//2]:.2f}s, "
          f"last @ {chunk_times[-1]:.2f}s")

In [ ]:
# ============================================================
# 14. Benchmark — throughput + concurrency for the selected model
# ============================================================
# vLLM's win over reference SDKs is CONTINUOUS BATCHING: many TTS requests
# share one engine via prefix-cached KV. This benchmark measures:
#   A) Serial latency — single request RTF (real-time factor).
#   B) Concurrency — fire N=8 at once; vLLM batches them -> wall ≈ one request.
#   C) Throughput — characters/sec across the batch.
# Results are written to /content/benchmark_<MODEL_KEY>.log (one file per model)
# so you can compare across models after running each.
import requests, time, concurrent.futures as cf, wave, io as _io, os

cfg = MODELS[MODEL_KEY]
BASE = f"http://localhost:{PORT}/v1/audio/speech"
SENTENCES = [
    "Xin chào, đây là giọng nói tiếng Việt từ {}.".format(MODEL_KEY),
    "Hôm nay thời tiết rất đẹp, trời trong xanh và mát mẻ.",
    "Trí tuệ nhân tạo đang thay đổi cách chúng ta tương tác với máy tính.",
    "Việt Nam là quốc gia đông dân nhất ở khu vực Đông Nam Á lục địa.",
    "Mô hình ngôn ngữ lớn có khả năng hiểu và tạo ra văn bản tự nhiên.",
    "Tổng hợp giọng nói là công nghệ biến văn bản thành âm thanh con người.",
    "Xin chào thế giới, chúc mọi người một ngày làm việc thật hiệu quả.",
    "Công nghệ mã nguồn mở giúp cộng đồng cùng phát triển và đổi mới.",
]

def _wav_dur(content):
    with wave.open(_io.BytesIO(content)) as w:
        return w.getnframes() / w.getframerate()

def synth(text, voice=None):
    if voice is None:
        voice = cfg["voice"]
    t = time.perf_counter()
    r = requests.post(BASE, json={"model": cfg["hf"], "input": text,
                                  "voice": voice,
                                  "response_format": cfg["response_format"]},
                      timeout=180)
    dt = time.perf_counter() - t
    if r.status_code != 200:
        raise RuntimeError(f"synth failed: {r.status_code} {r.text[:300]}")
    return r, dt, len(r.content), _wav_dur(r.content)

# --- A. Serial (cold + warm) ---
print("="*64)
print(f"A. SERIAL LATENCY  [{MODEL_KEY}]  (1 request, cold then warm)")
print("="*64)
r, cold_s, cold_bytes, cold_dur = synth(SENTENCES[0])
print(f"  cold : {cold_s:5.2f}s wall → {cold_dur:5.2f}s audio | "
      f"RTF={cold_s/cold_dur:.2f}x ({'faster' if cold_s<cold_dur else 'slower'} than realtime) "
      f"| {cold_bytes} bytes")

r, warm_s, warm_bytes, warm_dur = synth(SENTENCES[0])
print(f"  warm : {warm_s:5.2f}s wall → {warm_dur:5.2f}s audio | "
      f"RTF={warm_s/warm_dur:.2f}x (prefix cache hit — same ref voice, KV reused) "
      f"| {warm_bytes} bytes")

# --- B. Concurrency (the vLLM advantage) ---
N = 8
print("\n" + "="*64)
print(f"B. CONCURRENCY [{MODEL_KEY}]  ({N} requests fired simultaneously)")
print("="*64)
t0 = time.perf_counter()
with cf.ThreadPoolExecutor(N) as pool:
    futs = [pool.submit(synth, s) for s in SENTENCES[:N]]
    results = [f.result() for f in futs]
wall = time.perf_counter() - t0
total_audio = sum(res[3] for res in results)
total_chars = sum(len(s) for s in SENTENCES[:N])
print(f"  wall-clock : {wall:5.2f}s for {N} requests")
print(f"  total audio: {total_audio:5.2f}s")
print(f"  RTF (agg)  : {wall/total_audio:.2f}x  ({total_audio/wall:.1f}x realtime)")
print(f"  throughput : {total_chars/wall:6.0f} chars/s | {N/wall:.2f} req/s")
print(f"  per-req avg: {wall/N:5.2f}s (vs {warm_s:.2f}s serial warm)")
speedup = (warm_s * N) / wall
print(f"\n  ★ vLLM batching speedup vs serial: {speedup:.2f}x  "
      f"(serial would take ~{warm_s*N:.1f}s; vLLM did {wall:.1f}s)")

# --- C. Summary table ---
print("\n" + "="*64)
print(f"C. SUMMARY  [{MODEL_KEY}]")
print("="*64)
print(f"{'metric':<30}{'value':>14}")
print(f"{'-'*44}")
print(f"{'cold serial RTF':<30}{cold_s/cold_dur:>13.2f}x")
print(f"{'warm serial RTF':<30}{warm_s/warm_dur:>13.2f}x")
print(f"{'concurrent '+str(N)+' req wall':<30}{wall:>13.2f}s")
print(f"{'concurrent RTF (agg)':<30}{wall/total_audio:>13.2f}x")
print(f"{'concurrent throughput':<30}{total_chars/wall:>13.0f} c/s")
print(f"{'batching speedup vs serial':<30}{speedup:>13.2f}x")

# --- D. Write per-model benchmark log ---
log_path = f"/content/benchmark_{MODEL_KEY.replace(' ', '_').replace('-', '_')}.log"
with open(log_path, "w") as f:
    f.write(f"# {MODEL_KEY}  ({cfg['hf']})\n")
    f.write(f"voice={cfg['voice']}  sr={cfg['sr']}  dtype={cfg['dtype']}  "
            f"fmt={cfg['response_format']}\n\n")
    f.write(f"cold serial:  {cold_s:.2f}s wall / {cold_dur:.2f}s audio  RTF={cold_s/cold_dur:.2f}x\n")
    f.write(f"warm serial:  {warm_s:.2f}s wall / {warm_dur:.2f}s audio  RTF={warm_s/warm_dur:.2f}x\n")
    f.write(f"concurrent {N} wall: {wall:.2f}s\n")
    f.write(f"concurrent RTF (agg): {wall/total_audio:.2f}x  ({total_audio/wall:.1f}x realtime)\n")
    f.write(f"concurrent throughput: {total_chars/wall:.0f} chars/s  ({N/wall:.2f} req/s)\n")
    f.write(f"batching speedup vs serial: {speedup:.2f}x\n")
print(f"\n✓ logged → {log_path}")

In [ ]:
# ============================================================
# 12. (Optional) Voice cloning from ref_audio + ref_text
# ============================================================
# Upload a 1-30s Vietnamese/English clip + its transcript, then uncomment:
#
# from google.colab import files
# up = files.upload()
# ref_path = list(up.keys())[0]
# import base64, requests
# with open(ref_path, "rb") as f:
#     b64 = base64.b64encode(f.read()).decode()
# r = requests.post(f"http://localhost:{PORT}/v1/audio/speech", json={
#     "model": MODEL,
#     "input": "Hãy nói bằng giọng của tôi.",
#     "ref_audio": f"data:audio/wav;base64,{b64}",
#     "ref_text": "<exact transcript of the clip>",
# }, timeout=180)
# if r.status_code == 200:
#     open("/content/vieneu_clone.wav", "wb").write(r.content)
#     print("✓ wrote /content/vieneu_clone.wav", len(r.content), "bytes")
#     from IPython.display import Audio, display
#     display(Audio("/content/vieneu_clone.wav", autoplay=False))
# else:
#     print("✗ clone failed:", r.status_code, r.text[:1500])
